# Train D4-ORQB Model I from scratch

This notebook launches the clean two-stage training pipeline: classical-context backbone pretraining followed by the 40-epoch quantum stage. It contains no saved result, historical metric, or checkpoint. Dataset and storage paths are intentionally blank.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

import torch
import torchquantum as tq


def find_repo_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "d4_orqb" / "main.py").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the deeplense-quantum repository")


REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
print({"repository": str(REPO_ROOT), "cuda": torch.cuda.is_available(), "torchquantum": tq.__version__})

## Runtime paths

Fill all three paths on the GPU machine. `DEVELOPMENT_ROOT` must contain `axion/`, `cdm/`, and `no_sub/`. Use a new `RUN_NAME` for every attempt.

In [ ]:
DEVELOPMENT_ROOT = ""
CACHE_ROOT = ""
OUTPUT_ROOT = ""
RUN_NAME = "model_i_d4_orqb_run_01"

In [ ]:
EXPECTED_CLASSES = {"axion", "cdm", "no_sub"}


def required_path(raw, name):
    if not isinstance(raw, str) or not raw.strip():
        raise ValueError(f"Fill {name} before running")
    return Path(raw).expanduser().resolve()


DEVELOPMENT_PATH = required_path(DEVELOPMENT_ROOT, "DEVELOPMENT_ROOT")
CACHE_PATH = required_path(CACHE_ROOT, "CACHE_ROOT")
OUTPUT_ROOT_PATH = required_path(OUTPUT_ROOT, "OUTPUT_ROOT")
OUTPUT_DIR = OUTPUT_ROOT_PATH / RUN_NAME

if not DEVELOPMENT_PATH.is_dir():
    raise FileNotFoundError(f"Development dataset not found: {DEVELOPMENT_PATH}")
classes = {path.name for path in DEVELOPMENT_PATH.iterdir() if path.is_dir()}
if classes != EXPECTED_CLASSES:
    raise RuntimeError(f"Dataset classes are {sorted(classes)}; expected {sorted(EXPECTED_CLASSES)}")
missing = [name for name in sorted(EXPECTED_CLASSES) if not any((DEVELOPMENT_PATH / name).glob("*.npy"))]
if missing:
    raise RuntimeError(f"Class directories contain no .npy files: {missing}")
if not torch.cuda.is_available():
    raise RuntimeError("D4-ORQB training requires a CUDA-capable GPU")
if OUTPUT_DIR.exists():
    raise FileExistsError(f"Refusing to reuse output directory: {OUTPUT_DIR}")

CACHE_PATH.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT_PATH.mkdir(parents=True, exist_ok=True)
print({"development": str(DEVELOPMENT_PATH), "cache": str(CACHE_PATH), "output": str(OUTPUT_DIR), "gpu": torch.cuda.get_device_name(0)})

## Run both training stages

`--stage all` first trains the context-pretraining model, then initializes the selected shared components in the TorchQuantum model and runs the 40-epoch stage. The official test set is not opened.

In [ ]:
COMMAND = [
    sys.executable, "-m", "d4_orqb.main",
    "--development-root", str(DEVELOPMENT_PATH),
    "--cache-root", str(CACHE_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--stage", "all",
]
print("PYTHONPATH=src", shlex.join(COMMAND))

In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(SRC_ROOT) + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")
env["PYTHONDONTWRITEBYTECODE"] = "1"
env["PYTHONUNBUFFERED"] = "1"
subprocess.run(COMMAND, cwd=REPO_ROOT, env=env, check=True)

The fresh checkpoints, metrics, and other generated files remain in `OUTPUT_DIR` and are ignored by Git. Review that run before deliberately promoting any artifact.